# 02 Carry Research — Macro Metals System

> **Strategy:** Carry / Term-Structure (Memory File §3.2)
> **Scope:** In-sample development (2015–2022)
> **Sub-modules:** FX Cross-Sectional Carry · Metals Calendar Spreads · SOFR Curve
> **Integration:** Carry-with-trend filter from canonical TSMOM sleeve
>
> **How carry differs from TSMOM:**
> - TSMOM is *time-series*: sign(own 12M return) per instrument.
> - Carry is *cross-sectional* (rank instruments by implied yield) and
>   *term-structure* (calendar spreads, curve slope).
> - Carry and momentum are complementary premia with low correlation,
>   making them natural portfolio partners.
>
> Self-contained BQuant notebook — executable top-to-bottom.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import yaml
from pathlib import Path
from datetime import datetime
from typing import Dict, Tuple, Optional
from math import sqrt

# Bloomberg BQL
import bql
bq = bql.Service()

print(f"Session started : {datetime.now():%Y-%m-%d %H:%M}")
print(f"BQL service     : {type(bq).__name__}")
print(f"NumPy {np.__version__}  |  pandas {pd.__version__}")

## Config & Parameters

Load carry parameters from `parameters.yaml` (`strategies.carry` section).
Three sub-modules with distinct signal logic but common rebalance (weekly):

| Sub-module | Signal | Key params |
|------------|--------|------------|
| FX Carry | Cross-sectional rank of 3M implied carry | top/bot 30%, VIX crisis 30, trend filter |
| Metals Spreads | Z-score hysteresis on front−back spreads | 120d z, entry ±1.5σ, exit ±0.5σ |
| SOFR Curve | Slope z-scores + butterfly | 60d z, 10bp threshold, policy stability |

All sizing frozen at weekly rebalance (W-FRI) and forward-filled.

In [ ]:
CONFIG_DIR = Path("config")

with open(CONFIG_DIR / "parameters.yaml") as f:
    params = yaml.safe_load(f)
with open(CONFIG_DIR / "tickers.yaml") as f:
    tickers = yaml.safe_load(f)

gcfg       = params["global"]
carry_cfg  = params["strategies"]["carry"]
targets    = params["performance_targets"]

fx_cfg   = carry_cfg["fx"]
met_cfg  = carry_cfg["metals"]["calendar_spreads"]
sofr_cfg = carry_cfg["sofr_curve"]

IS_START = gcfg["in_sample_start"]
IS_END   = gcfg["in_sample_end"]

# Common carry parameters
TARGET_VOL       = gcfg.get("target_portfolio_vol_annual", 0.10)
VOL_LAMBDA       = gcfg.get("vol_decay_lambda", 0.94)
LEV_CAP          = gcfg.get("vol_cap_multiplier", 2.0)
TC_BP            = 2.0
TREND_FILTER_MODE = "soft"   # "hard" = zero weight, "soft" = 0.5x weight

# All submodules rebalance weekly (frozen within week)
REBAL_FREQ       = "W-FRI"

# Spread sign convention (Part B)
SPREAD_CONVENTION = met_cfg.get("spread_definition", "FminusB")

# SOFR inactive redistribution
REDISTRIBUTE_INACTIVE = carry_cfg.get("redistribute_inactive_module", True)

print("Carry parameters:")
print(f"\n  FX Carry:")
print(f"    Pairs            : {fx_cfg['pairs']}")
print(f"    Horizon          : {fx_cfg['carry_horizon_days']}d")
print(f"    Rank top/bot     : {fx_cfg['ranking_top_pct']:.0%} / {fx_cfg['ranking_bottom_pct']:.0%}")
print(f"    Min carry        : {fx_cfg['min_carry_bp']} bp")
print(f"    Trend filter     : {fx_cfg['trend_filter_lookback_days']}d")
print(f"    Crisis VIX       : {fx_cfg['crisis_vix_threshold']}")

print(f"\n  Metals Calendar Spreads:")
print(f"    Instruments      : {met_cfg['instruments']}")
print(f"    Z-score window   : {met_cfg['zscore_lookback_days']}d")
print(f"    Entry/exit z     : ±{met_cfg['entry_zscore']} / ±{met_cfg['exit_zscore']}")
print(f"    Spread convention: {SPREAD_CONVENTION}")

print(f"\n  SOFR Curve:")
print(f"    Instruments      : {sofr_cfg['instruments']}")
print(f"    Z-score window   : {sofr_cfg['zscore_lookback_days']}d")
print(f"    Slope thresh.    : {sofr_cfg['steepness_threshold_bp']} bp")

print(f"\n  Common:")
print(f"    Vol target       : {TARGET_VOL:.0%}")
print(f"    EWMA lambda      : {VOL_LAMBDA}")
print(f"    TC per side      : {TC_BP:.1f} bp")
print(f"    Trend filter     : {TREND_FILTER_MODE}")
print(f"    Rebalance        : weekly ({REBAL_FREQ})")
print(f"    IS period        : {IS_START} to {IS_END}")
print(f"    Redistribute     : {REDISTRIBUTE_INACTIVE}")

## Data Loader (BQL)

`BQuantDataLoader` with methods for:
- Single ticker history
- Multi-ticker panel (curve points, spread components)
- FX spot + forward pair

Uses corrected BQL syntax: `df.set_index('DATE')`, dedup, `pd.to_datetime`.

In [ ]:
class BQuantDataLoader:
    """Fetch historical prices via Bloomberg BQL."""

    def __init__(self, ticker_map: dict) -> None:
        self._tickers = ticker_map
        self._bq = bql.Service()

    def _resolve(self, logical_name: str) -> Optional[str]:
        """Logical name -> Bloomberg ticker string, or None."""
        for group in self._tickers.values():
            if isinstance(group, dict) and logical_name in group:
                return group[logical_name]
        return None

    def get_history(self, logical_name: str, start: str, end: str,
                    field: str = "PX_LAST") -> pd.Series:
        """Fetch daily price series for one instrument."""
        bbg = self._resolve(logical_name)
        if bbg is None:
            print(f"  ! {logical_name}: not in tickers.yaml")
            return pd.Series(dtype=float, name=logical_name)
        request = bql.Request(
            bbg,
            {field: self._bq.data.px_last(
                dates=self._bq.func.range(start, end)
            )},
        )
        try:
            response = self._bq.execute(request)
            df = response[0].df()
            if df.empty:
                print(f"  ! {logical_name}: empty response")
                return pd.Series(dtype=float, name=logical_name)
            df_fixed = df.set_index('DATE')
            series = df_fixed[field]
            series.index = pd.to_datetime(series.index, errors='coerce')
            series = series.dropna()
            series = series[~series.index.duplicated(keep='last')]
            series = series.sort_index().astype(float)
            series.name = logical_name
            series.index.name = "date"
            return series
        except Exception as exc:
            print(f"  ! {logical_name} ({bbg}): {exc}")
            return pd.Series(dtype=float, name=logical_name)

    def get_multi_history(self, logical_names: list, start: str, end: str,
                          field: str = "PX_LAST") -> pd.DataFrame:
        """Fetch panel of prices for multiple tickers."""
        panel = {}
        for name in logical_names:
            s = self.get_history(name, start, end, field)
            if len(s) > 0:
                panel[name] = s
        df = pd.DataFrame(panel).sort_index()
        df = df[df.index.notna()].ffill()
        df = df.dropna(axis=1, how="all")
        return df

    def get_fx_spot_and_fwd(self, spot_name: str, fwd_name: str,
                            start: str, end: str) -> pd.DataFrame:
        """Fetch FX spot + 3M forward outright."""
        spot = self.get_history(spot_name, start, end)
        fwd  = self.get_history(fwd_name, start, end)
        df = pd.DataFrame({"spot": spot, "forward": fwd}).dropna()
        df["fwd_points"] = df["forward"] - df["spot"]
        return df

    def get_curve_points(self, logical_names: list, start: str,
                         end: str) -> pd.DataFrame:
        """Alias for get_multi_history (curve context)."""
        return self.get_multi_history(logical_names, start, end)


loader = BQuantDataLoader(tickers)
print("BQuantDataLoader ready.")

## Carry Universe & Breadth Report

Three sub-universes:
1. **FX Carry:** G10 pairs with 3M forwards available
2. **Metals Spreads:** GC and SI front-3 calendar spreads
3. **SOFR Curve:** Front 4 quarterly SOFR futures

In [ ]:
# ── FX Carry: spot + forward pairs ────────────────────────────────
FX_PAIRS = {
    "eurusd": ("eurusd_spot", "eurusd_3m_fwd"),
    "usdjpy": ("usdjpy_spot", "usdjpy_3m_fwd"),
    "gbpusd": ("gbpusd_spot", "gbpusd_3m_fwd"),
    "audusd": ("audusd_spot", "audusd_3m_fwd"),
    "usdcnh": ("usdcnh_spot", "usdcnh_3m_fwd"),
}

FX_PAIR_MAP = {
    "eurusd_spot": "eurusd", "usdjpy_spot": "usdjpy",
    "gbpusd_spot": "gbpusd", "audusd_spot": "audusd",
    "usdchf_spot": None,     "usdcnh_spot": "usdcnh",
}

print("=== FX Carry Data ===")
fx_data = {}
for pair_cfg in fx_cfg["pairs"]:
    key = FX_PAIR_MAP.get(pair_cfg)
    if key is None:
        print(f"  {pair_cfg:18s} SKIP (no forward ticker)")
        continue
    spot_name, fwd_name = FX_PAIRS[key]
    print(f"  {key:18s}", end=" ")
    try:
        df = loader.get_fx_spot_and_fwd(spot_name, fwd_name, IS_START, IS_END)
        fx_data[key] = df
        print(f"OK  {len(df):>5d} obs  [{df.index[0]:%Y-%m-%d} -> {df.index[-1]:%Y-%m-%d}]")
    except Exception as e:
        print(f"FAIL: {e}")

fx_spot_df = pd.DataFrame({k: v["spot"] for k, v in fx_data.items()}).ffill()

# ── Metals futures (front/second/third for GC and SI) ─────────────
print("\n=== Metals Calendar Spread Data ===")
metals_tickers = met_cfg["instruments"]
metals_prices = loader.get_curve_points(metals_tickers, IS_START, IS_END)
print(f"  Panel: {metals_prices.shape[0]} days x {metals_prices.shape[1]} contracts")

# ── SOFR futures ──────────────────────────────────────────────────
print("\n=== SOFR Curve Data ===")
sofr_tickers = sofr_cfg["instruments"]
sofr_prices = loader.get_curve_points(sofr_tickers, IS_START, IS_END)
print(f"  Panel: {sofr_prices.shape[0]} days x {sofr_prices.shape[1]} contracts")

# ── VIX for crisis filter ────────────────────────────────────────
print("\n=== Risk Indicators ===")
vix = loader.get_history("vix", IS_START, IS_END)
print(f"  VIX: {len(vix)} obs")

# ── Breadth report ────────────────────────────────────────────────
print("\n" + "=" * 65)
print("  BREADTH REPORT")
print("=" * 65)
print(f"  FX Carry       : {len(fx_data)}/{len(FX_PAIRS)} pairs with spot+fwd")
print(f"  Metals Spreads : {metals_prices.shape[1]}/{len(metals_tickers)} contracts")
print(f"  SOFR Curve     : {sofr_prices.shape[1]}/{len(sofr_tickers)} contracts")
print(f"  VIX            : {'OK' if len(vix) > 0 else 'MISSING'}")

## Data Availability Gating (SOFR) & Coverage Report

Before running any submodule, verify that required series are present.
SOFR curve requires **≥ 2 contract series** with overlapping history;
if fewer are available, the module is set to INACTIVE and the combined
carry sleeve continues without it.

Export: `outputs/carry_data_coverage.csv`

In [ ]:
# ── Coverage report for ALL carry sub-modules ─────────────────────
output_dir = Path("../outputs")
output_dir.mkdir(exist_ok=True)

coverage_rows = []

# FX
for pair, df in fx_data.items():
    coverage_rows.append({
        "module": "FX Carry", "series": pair,
        "status": "available", "n_obs": len(df),
        "first_valid": str(df.index[0].date()) if len(df) > 0 else "N/A",
        "last_valid": str(df.index[-1].date()) if len(df) > 0 else "N/A",
    })

# Metals
for col in metals_tickers:
    available = col in metals_prices.columns
    if available:
        s = metals_prices[col].dropna()
        coverage_rows.append({
            "module": "Metals Spreads", "series": col,
            "status": "available", "n_obs": len(s),
            "first_valid": str(s.index[0].date()) if len(s) > 0 else "N/A",
            "last_valid": str(s.index[-1].date()) if len(s) > 0 else "N/A",
        })
    else:
        coverage_rows.append({
            "module": "Metals Spreads", "series": col,
            "status": "MISSING", "n_obs": 0,
            "first_valid": "N/A", "last_valid": "N/A",
        })

# SOFR — explicit gating
sofr_available = [c for c in sofr_tickers if c in sofr_prices.columns]
SOFR_MODULE_ACTIVE = len(sofr_available) >= 2

for col in sofr_tickers:
    available = col in sofr_prices.columns
    if available:
        s = sofr_prices[col].dropna()
        coverage_rows.append({
            "module": "SOFR Curve", "series": col,
            "status": "available", "n_obs": len(s),
            "first_valid": str(s.index[0].date()) if len(s) > 0 else "N/A",
            "last_valid": str(s.index[-1].date()) if len(s) > 0 else "N/A",
        })
    else:
        coverage_rows.append({
            "module": "SOFR Curve", "series": col,
            "status": "MISSING", "n_obs": 0,
            "first_valid": "N/A", "last_valid": "N/A",
        })

coverage_df = pd.DataFrame(coverage_rows)
coverage_df.to_csv(output_dir / "carry_data_coverage.csv", index=False)

print("=" * 70)
print("  DATA COVERAGE REPORT")
print("=" * 70)
display(coverage_df)

print(f"\n  SOFR module status: ", end="")
if SOFR_MODULE_ACTIVE:
    # Compute active date range
    sofr_first = sofr_prices[sofr_available].apply(lambda s: s.dropna().index[0]
                                                    if s.dropna().shape[0] > 0
                                                    else pd.NaT).max()
    sofr_last = sofr_prices[sofr_available].apply(lambda s: s.dropna().index[-1]
                                                   if s.dropna().shape[0] > 0
                                                   else pd.NaT).min()
    print(f"ACTIVE ({len(sofr_available)}/{len(sofr_tickers)} tenors)")
    print(f"    Active date range: {sofr_first} to {sofr_last}")
else:
    print(f"INACTIVE (insufficient tenors: {len(sofr_available)}/{len(sofr_tickers)})")
    print(f"    Required: >= 2 series  |  Available: {sofr_available}")
    print(f"    SOFR curve will be SKIPPED. Combined sleeve continues without it.")

print(f"\n  Exported: carry_data_coverage.csv")

## Metals Spreads — Convention Check (Front−Back vs Back−Front)

Standard terminology:
- **Backwardation** = near-month priced higher than deferred → `Front − Back > 0`
- **Contango** = deferred priced higher than near → `Front − Back < 0`

We verify both definitions are exact negatives and print frequency tables.
The convention used for the rest of the notebook is controlled by
`params['carry']['metals_spreads'].get('spread_definition', 'FminusB')`.

Export: `outputs/debug_spread_convention_sample.csv`

In [ ]:
spread_pair_defs = {
    "GC_1v2": ("gc_fut_front", "gc_fut_second"),
    "GC_2v3": ("gc_fut_second", "gc_fut_third"),
    "SI_1v2": ("si_fut_front", "si_fut_second"),
    "SI_2v3": ("si_fut_second", "si_fut_third"),
}

print("=" * 70)
print("  SPREAD SIGN CONVENTION CHECK")
print("=" * 70)

convention_rows = []

for label, (front_name, back_name) in spread_pair_defs.items():
    if front_name not in metals_prices.columns or back_name not in metals_prices.columns:
        print(f"  {label}: SKIP (missing data)")
        continue

    front = metals_prices[front_name]
    back  = metals_prices[back_name]

    spread_FmB = front - back    # front minus back
    spread_BmF = back - front    # back minus front

    # Assert they are negatives of each other
    max_diff = (spread_FmB + spread_BmF).abs().max()
    assert max_diff < 1e-8, f"{label}: FmB + BmF != 0 (max diff = {max_diff})"

    n_valid = spread_FmB.dropna().shape[0]
    if n_valid == 0:
        continue

    backwd_pct = (spread_FmB.dropna() > 0).sum() / n_valid * 100
    contango_pct = (spread_FmB.dropna() < 0).sum() / n_valid * 100
    flat_pct = (spread_FmB.dropna() == 0).sum() / n_valid * 100

    print(f"\n  {label} (Front={front_name}, Back={back_name}):")
    print(f"    FmB definition: backwardation={backwd_pct:.1f}%  contango={contango_pct:.1f}%  flat={flat_pct:.1f}%")
    print(f"    BmF definition: backwardation={contango_pct:.1f}%  contango={backwd_pct:.1f}%  flat={flat_pct:.1f}%")
    print(f"    FmB+BmF max abs diff = {max_diff:.2e}  (OK)")

    convention_rows.append({
        "spread": label, "front": front_name, "back": back_name,
        "backwd_pct_FmB": round(backwd_pct, 1),
        "contango_pct_FmB": round(contango_pct, 1),
    })

print(f"\n  Using spread definition: {SPREAD_CONVENTION}")

# Build spread series using chosen convention
metals_spreads = pd.DataFrame(index=metals_prices.index)
for label, (front_name, back_name) in spread_pair_defs.items():
    if front_name not in metals_prices.columns or back_name not in metals_prices.columns:
        continue
    if SPREAD_CONVENTION == "BminusF":
        metals_spreads[label] = metals_prices[back_name] - metals_prices[front_name]
    else:  # default FminusB
        metals_spreads[label] = metals_prices[front_name] - metals_prices[back_name]

# Export sample
sample = metals_spreads.dropna()
if len(sample) > 0:
    sample_out = pd.concat([sample.head(10), sample.tail(10)])
    sample_out.to_csv(output_dir / "debug_spread_convention_sample.csv")
    print(f"  Exported: debug_spread_convention_sample.csv ({len(sample_out)} rows)")

## TSMOM Trend Filter Integration

Import the canonical TSMOM monthly sign signal to use as a **carry filter**:
- If TSMOM file exists (`outputs/tsmom_signals_monthly_*.csv`), load it.
- Otherwise, compute minimal 12M sign filter for carry underlyings.
- Trend filter mode: **soft** (scale carry weight by 0.5 when misaligned) or
  **hard** (zero out misaligned carry positions).

In [ ]:
def load_or_compute_trend_filter(
    prices_df: pd.DataFrame,
    output_dir: Path = Path("../outputs"),
    lookback: int = 252,
) -> pd.DataFrame:
    """Load TSMOM signals or compute minimal 12M sign filter.

    Returns DataFrame of daily signals in {-1, 0, +1}.
    """
    import glob
    pattern = str(output_dir / "tsmom_signals_monthly_*.csv")
    files = sorted(glob.glob(pattern))
    if files:
        latest = files[-1]
        print(f"  Loaded TSMOM signals from: {Path(latest).name}")
        sig = pd.read_csv(latest, index_col=0, parse_dates=True)
        sig = sig.reindex(prices_df.index).ffill().fillna(0.0)
        return sig

    # Fallback: compute minimal 12M sign, monthly frozen
    print("  TSMOM export not found — computing minimal trend filter")
    r12 = prices_df.pct_change(lookback)
    sig_daily = np.sign(r12).shift(1)
    sig_monthly = sig_daily.resample("M").last().shift(1)
    sig = sig_monthly.reindex(prices_df.index).ffill().fillna(0.0)
    return sig


# Build combined price panel for trend filter
trend_prices = {}
for pair, df in fx_data.items():
    trend_prices[pair] = df["spot"]
for inst in ["gc_fut_front", "si_fut_front"]:
    if inst in metals_prices.columns:
        trend_prices[inst] = metals_prices[inst]
if "sofr_fut_front" in sofr_prices.columns:
    trend_prices["sofr_fut_front"] = sofr_prices["sofr_fut_front"]

trend_prices_df = pd.DataFrame(trend_prices).sort_index().ffill()
trend_filter_daily = load_or_compute_trend_filter(trend_prices_df)

print(f"\n  Trend filter shape: {trend_filter_daily.shape}")
print(f"  Columns: {list(trend_filter_daily.columns)}")

## Carry Strategy

Three submodules with a common interface:
- **A) FX Carry:** Cross-sectional ranking of implied carry, trend-filtered, weekly frozen
- **B) Metals Spreads:** Z-score entry/exit with hysteresis, trend-scaled
- **C) SOFR Curve:** Slope z-scores, butterfly, weekly frozen (with availability gating)

All submodules:
- Freeze weights at weekly rebalance dates (no intra-week resizing)
- Vol-scale using EWMA(λ=0.94) sampled at rebalance and frozen
- Apply trend filter (soft: 0.5× when misaligned)

In [ ]:
def ewma_vol_ann(returns: pd.Series, lam: float = 0.94) -> pd.Series:
    """EWMA annualised volatility (RiskMetrics)."""
    alpha = 1 - lam
    ewma_var = returns.pow(2).ewm(alpha=alpha, adjust=False).mean()
    return np.sqrt(ewma_var) * np.sqrt(252)


def freeze_weekly(series: pd.DataFrame, freq: str = "W-FRI") -> pd.DataFrame:
    """Sample at weekly frequency and forward-fill (freeze within week)."""
    weekly = series.resample(freq).last()
    return weekly.reindex(series.index).ffill()


def apply_trend_filter(
    weights: pd.DataFrame,
    trend: pd.DataFrame,
    col_map: dict,
    mode: str = "soft",
) -> pd.DataFrame:
    """Apply TSMOM trend filter to carry weights.

    col_map: maps weight column -> trend column name.
    mode: "soft" (0.5x when misaligned) or "hard" (0x).
    Returns (filtered_weights, diagnostics_dict).
    """
    filtered = weights.copy()
    scale = 0.0 if mode == "hard" else 0.5
    diag = {}
    for w_col, t_col in col_map.items():
        if w_col not in filtered.columns or t_col not in trend.columns:
            diag[w_col] = {"mapped": False, "pct_modified": 0.0}
            continue
        t = trend[t_col].reindex(filtered.index).ffill().fillna(0)
        # Misaligned: carry weight sign != trend sign (and both non-zero)
        w_nz = np.sign(filtered[w_col]) != 0
        t_nz = np.sign(t) != 0
        misaligned = w_nz & t_nz & (np.sign(filtered[w_col]) != np.sign(t))
        n_active = (w_nz & t_nz).sum()
        n_misaligned = misaligned.sum()
        pct_mod = n_misaligned / max(n_active, 1) * 100
        diag[w_col] = {"mapped": True, "pct_modified": pct_mod,
                        "n_active": int(n_active), "n_misaligned": int(n_misaligned)}
        filtered.loc[misaligned, w_col] *= scale
    return filtered, diag


class CarryStrategy:
    """Carry / term-structure strategy (§3.2) with 3 sub-modules."""

    def __init__(self, params: dict, tgt_vol: float, vol_lam: float,
                 lev_cap: float, rebal_freq: str) -> None:
        self.fx_cfg   = params["strategies"]["carry"]["fx"]
        self.met_cfg  = params["strategies"]["carry"]["metals"]["calendar_spreads"]
        self.sofr_cfg = params["strategies"]["carry"]["sofr_curve"]
        self.tgt_vol  = tgt_vol
        self.vol_lam  = vol_lam
        self.lev_cap  = lev_cap
        self.rebal    = rebal_freq

    # ══════════════════════════════════════════════════════════════
    # A) FX CARRY (cross-sectional)
    # ══════════════════════════════════════════════════════════════
    def fx_carry_signals(
        self, fx_data: dict, vix: pd.Series,
    ) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
        """Generate FX carry signals and vol-scaled weights."""
        horizon = self.fx_cfg["carry_horizon_days"]
        top_pct = self.fx_cfg["ranking_top_pct"]
        bot_pct = self.fx_cfg["ranking_bottom_pct"]
        min_bp  = self.fx_cfg["min_carry_bp"]
        vix_thr = self.fx_cfg["crisis_vix_threshold"]
        use_crisis = self.fx_cfg.get("crisis_filter", True)

        carry_df = pd.DataFrame()
        spot_df  = pd.DataFrame()
        for pair, df in fx_data.items():
            implied = (-df["fwd_points"] / df["spot"]) * (365 / horizon)
            carry_df[pair] = implied
            spot_df[pair]  = df["spot"]

        idx = carry_df.dropna(how="all").index
        carry_df = carry_df.reindex(idx)
        spot_df  = spot_df.reindex(idx)
        vix_a    = vix.reindex(idx).ffill()

        # Cross-sectional ranking
        ranks = carry_df.rank(axis=1, pct=True)
        signals = pd.DataFrame(0.0, index=idx, columns=carry_df.columns)
        for col in carry_df.columns:
            sig = pd.Series(0.0, index=idx)
            sig[ranks[col] >= (1 - top_pct)] =  1.0
            sig[ranks[col] <= bot_pct]        = -1.0
            sig[carry_df[col].abs() * 10_000 < min_bp] = 0.0
            signals[col] = sig

        # Crisis filter
        if use_crisis:
            crisis = vix_a > vix_thr
            signals.loc[crisis] = 0.0

        # Freeze weekly — SIGNALS frozen at rebalance
        signals = freeze_weekly(signals, self.rebal)

        # Vol-scale: EWMA on spot returns, sampled weekly and frozen
        spot_ret = spot_df.pct_change().fillna(0.0)
        vol_ann = pd.DataFrame({
            col: ewma_vol_ann(spot_ret[col], self.vol_lam) for col in spot_df.columns
        })
        vol_weekly = freeze_weekly(vol_ann, self.rebal)
        vol_weekly = vol_weekly.replace(0, np.nan)

        weights = signals * (self.tgt_vol / vol_weekly)
        cap = self.lev_cap * (self.tgt_vol / vol_weekly)
        weights = weights.clip(-cap, cap).fillna(0.0)

        # FINAL freeze: weights should only change on rebalance dates
        weights = freeze_weekly(weights, self.rebal)

        return signals, weights, carry_df

    # ══════════════════════════════════════════════════════════════
    # B) METALS CALENDAR SPREADS
    # ══════════════════════════════════════════════════════════════
    def metals_spread_signals(
        self, spreads_df: pd.DataFrame,
    ) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
        """Generate metals calendar spread z-score signals.

        spreads_df: pre-computed spread series (using chosen convention).
        Returns: (signals, weights, zscores)
        """
        zs_lb   = self.met_cfg["zscore_lookback_days"]
        entry_z = self.met_cfg["entry_zscore"]
        exit_z  = self.met_cfg["exit_zscore"]

        signals  = pd.DataFrame(index=spreads_df.index)
        zscores  = pd.DataFrame(index=spreads_df.index)

        for label in spreads_df.columns:
            spread = spreads_df[label].copy()

            # Rolling z-score
            mu  = spread.rolling(zs_lb, min_periods=zs_lb // 2).mean()
            sig = spread.rolling(zs_lb, min_periods=zs_lb // 2).std().replace(0, np.nan)
            z   = (spread - mu) / sig
            zscores[label] = z

            # Hysteresis signal
            signal = pd.Series(0.0, index=spreads_df.index)
            pos = 0.0
            for i in range(len(z)):
                if pd.isna(z.iloc[i]):
                    signal.iloc[i] = 0.0
                    continue
                zv = z.iloc[i]
                if pos == 0.0:
                    if zv > entry_z:
                        pos = -1.0   # mean-revert: sell rich spread
                    elif zv < -entry_z:
                        pos = 1.0    # mean-revert: buy cheap spread
                elif pos > 0 and zv > -exit_z:
                    pos = 0.0
                elif pos < 0 and zv < exit_z:
                    pos = 0.0
                signal.iloc[i] = pos

            signals[label] = signal

        # Freeze weekly
        signals = freeze_weekly(signals, self.rebal)

        return signals, zscores

    # ══════════════════════════════════════════════════════════════
    # C) SOFR CURVE (with availability gating)
    # ══════════════════════════════════════════════════════════════
    def sofr_curve_signals(
        self, prices: pd.DataFrame, module_active: bool,
    ) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, str]:
        """Generate SOFR curve slope/butterfly signals.

        Returns: (signals, weights, slopes, status_string)
        """
        empty = pd.DataFrame(index=prices.index if len(prices) > 0
                             else pd.DatetimeIndex([]))

        if not module_active:
            return empty, empty, empty, "INACTIVE (insufficient tenors)"

        zs_lb     = self.sofr_cfg["zscore_lookback_days"]
        stability = self.sofr_cfg["policy_stability_window_days"]
        contracts = [c for c in self.sofr_cfg["instruments"] if c in prices.columns]

        if len(contracts) < 2:
            return empty, empty, empty, "INACTIVE (< 2 contracts in data)"

        slopes  = pd.DataFrame(index=prices.index)
        signals = pd.DataFrame(index=prices.index)

        # Adjacent slopes
        for i in range(len(contracts) - 1):
            f, b = contracts[i], contracts[i + 1]
            label = f"SOFR_{i+1}v{i+2}"
            slope = prices[b] - prices[f]
            slopes[label] = slope

            mu  = slope.rolling(zs_lb, min_periods=zs_lb // 2).mean()
            sig = slope.rolling(zs_lb, min_periods=zs_lb // 2).std().replace(0, np.nan)
            z = (slope - mu) / sig
            signal = np.tanh(z)

            # Policy stability filter
            slope_vol = slope.rolling(stability).std()
            sv_mu  = slope_vol.rolling(zs_lb).mean()
            sv_sig = slope_vol.rolling(zs_lb).std().replace(0, np.nan)
            sv_z   = (slope_vol - sv_mu) / sv_sig
            signal[sv_z.abs() > 2.0] *= 0.5

            signals[label] = signal

        # Butterfly (if 3+ contracts)
        if len(contracts) >= 3:
            fly = prices[contracts[0]] - 2 * prices[contracts[1]] + prices[contracts[2]]
            slopes["SOFR_fly"] = fly
            mu  = fly.rolling(zs_lb, min_periods=zs_lb // 2).mean()
            sig = fly.rolling(zs_lb, min_periods=zs_lb // 2).std().replace(0, np.nan)
            signals["SOFR_fly"] = np.tanh((fly - mu) / sig)

        # Freeze weekly
        signals = freeze_weekly(signals, self.rebal)

        status = f"ACTIVE ({len(contracts)} contracts, {len(signals.columns)} signals)"
        return signals, signals.copy(), slopes, status

    # ══════════════════════════════════════════════════════════════
    # run_all
    # ══════════════════════════════════════════════════════════════
    def run_all(self, fx_data, metals_spreads, sofr_prices, vix,
                sofr_active):
        """Run all 3 submodules, return structured results."""
        fx_sig, fx_w, fx_carry = self.fx_carry_signals(fx_data, vix)
        mt_sig, mt_z = self.metals_spread_signals(metals_spreads)
        sf_sig, sf_w, sf_slopes, sf_status = self.sofr_curve_signals(
            sofr_prices, sofr_active)
        return {
            "fx":    {"signals": fx_sig, "weights": fx_w, "carry_matrix": fx_carry},
            "metals":{"signals": mt_sig, "zscores": mt_z},
            "sofr":  {"signals": sf_sig, "weights": sf_w, "slopes": sf_slopes,
                      "status": sf_status},
        }


carry = CarryStrategy(params, TARGET_VOL, VOL_LAMBDA, LEV_CAP, REBAL_FREQ)
print("CarryStrategy initialised with 3 sub-modules.")

## Generate Signals (IS Period)

Run all three carry sub-modules and inspect signal diagnostics.
Includes trend filter wiring diagnostic (Part D.1).

In [ ]:
results = carry.run_all(fx_data, metals_spreads, sofr_prices, vix,
                       SOFR_MODULE_ACTIVE)

# Report SOFR status
print(f"SOFR Curve module status: {results['sofr']['status']}")

# ── Apply trend filter ───────────────────────────────────────────
fx_trend_map = {col: col for col in results["fx"]["weights"].columns
                if col in trend_filter_daily.columns}
metals_trend_map = {}
for col in results["metals"]["signals"].columns:
    if col.startswith("GC"):
        metals_trend_map[col] = "gc_fut_front"
    elif col.startswith("SI"):
        metals_trend_map[col] = "si_fut_front"
sofr_trend_map = {col: "sofr_fut_front" for col in results["sofr"]["signals"].columns
                  if "sofr_fut_front" in trend_filter_daily.columns}

# Pure carry weights
fx_w_pure = results["fx"]["weights"].copy()
mt_sig_pure = results["metals"]["signals"].copy()
sf_sig_pure = results["sofr"]["signals"].copy()

# Apply trend filter — returns (filtered, diag)
fx_w_trend, fx_trend_diag = apply_trend_filter(
    fx_w_pure, trend_filter_daily, fx_trend_map, TREND_FILTER_MODE)
mt_sig_trend, mt_trend_diag = apply_trend_filter(
    mt_sig_pure, trend_filter_daily, metals_trend_map, TREND_FILTER_MODE)
sf_sig_trend, sf_trend_diag = apply_trend_filter(
    sf_sig_pure, trend_filter_daily, sofr_trend_map, TREND_FILTER_MODE)

# ── Trend filter wiring diagnostic (Part D.1) ────────────────────
print("\n" + "=" * 70)
print("  TREND FILTER WIRING DIAGNOSTIC")
print("=" * 70)
for module_name, diag in [("FX Carry", fx_trend_diag),
                           ("Metals Spreads", mt_trend_diag),
                           ("SOFR Curve", sf_trend_diag)]:
    print(f"\n  {module_name} (mode={TREND_FILTER_MODE}):")
    if not diag:
        print("    No columns mapped.")
        continue
    for col, info in diag.items():
        if not info.get("mapped", False):
            print(f"    {col:18s}  NOT MAPPED (no trend column)")
        else:
            pct = info["pct_modified"]
            n_m = info["n_misaligned"]
            n_a = info["n_active"]
            print(f"    {col:18s}  {pct:5.1f}% modified  ({n_m}/{n_a} active days)")
            if pct < 1.0:
                print(f"      ** WARNING: Trend filter not active — check series alignment/quote convention.")
            elif TREND_FILTER_MODE == "soft":
                # For soft mode, compute average scaling factor
                avg_scale = 1.0 - (pct / 100.0) * 0.5
                print(f"      Average scaling factor: {avg_scale:.3f}")

# ── Signal diagnostics ───────────────────────────────────────────
print("\n" + "=" * 70)
print("  SIGNAL DIAGNOSTICS")
print("=" * 70)
for name, sigs in [("FX Carry", results["fx"]["signals"]),
                   ("Metals Spreads", results["metals"]["signals"]),
                   ("SOFR Curve", results["sofr"]["signals"])]:
    print(f"\n  {name}:")
    if sigs.empty:
        print("    No signals (module inactive)")
        continue
    for col in sigs.columns:
        s = sigs[col]
        n = len(s)
        pct_pos  = (s > 0).sum() / n * 100
        pct_neg  = (s < 0).sum() / n * 100
        pct_flat = (s == 0).sum() / n * 100
        changes  = (s.diff().abs() > 1e-10).sum()
        avg_hold = n / max(changes, 1)
        print(f"    {col:15s}  +={pct_pos:4.1f}%  -={pct_neg:4.1f}%  "
              f"flat={pct_flat:4.1f}%  changes={changes}  hold={avg_hold:.0f}d")

## Backtest Engines (carry-specific)

Two backtest variants:
- **`backtest_weights`**: For FX carry (weights × spot returns)
- **`backtest_spread`**: For metals spreads and SOFR slopes (PnL / capital approach)

**Part C fix:** `backtest_spread()` now uses an explicit position-units approach:
1. Compute spread price changes `dS` (in price units, NOT pct_change)
2. Estimate spread vol in price units (EWMA on `dS`)
3. Size position in spread units: `pos = signal × (vol_target × capital) / vol_ann`
4. PnL = `pos.shift(1) × dS`; returns = `PnL / capital`
5. Finite assertions — no inf/nan allowed

This eliminates the inf% bug caused by spreads crossing zero during pct_change.

In [ ]:
def backtest_weights(
    prices: pd.DataFrame,
    weights: pd.DataFrame,
    tc_bp: float = 2.0,
    capital: float = 1_000_000.0,
) -> pd.DataFrame:
    """Backtest using weights on price returns (FX carry)."""
    ret = prices.pct_change().fillna(0.0)
    common_cols = [c for c in weights.columns if c in ret.columns]
    w = weights[common_cols].reindex(ret.index).ffill().fillna(0.0)
    r = ret[common_cols]

    port_ret_gross = (w.shift(1) * r).sum(axis=1)
    turnover = (w - w.shift(1)).abs().sum(axis=1).fillna(0.0)
    tc = turnover * (tc_bp / 10_000)
    port_ret_net = port_ret_gross - tc
    equity = capital * (1 + port_ret_net).cumprod()

    return pd.DataFrame({
        "ret_gross": port_ret_gross, "ret_net": port_ret_net,
        "equity": equity, "turnover": turnover,
    })


def backtest_spread(
    spread_px: pd.DataFrame,
    signal: pd.DataFrame,
    vol_target_annual: float = 0.10,
    vol_lookback: int = 30,
    rebalance: str = "W-FRI",
    tc_bp_per_side: float = 2.0,
    capital_base: float = 1_000_000.0,
    name: str = "",
) -> dict:
    """Backtest spread trades using PnL/capital approach (Part C).

    Parameters
    ----------
    spread_px : DataFrame of spread price levels (one column per spread)
    signal : DataFrame of signals (same columns)
    vol_target_annual : annual vol target for position sizing
    vol_lookback : EWMA span for vol estimation
    rebalance : rebalance frequency string
    tc_bp_per_side : transaction cost per side in basis points
    capital_base : notional capital

    Returns
    -------
    dict with 'ret_net', 'equity', 'pos_units', 'turnover_units', 'cost',
         plus per-spread diagnostics
    """
    common_cols = [c for c in signal.columns if c in spread_px.columns]
    if not common_cols:
        return {"ret_net": pd.Series(dtype=float),
                "equity": pd.Series(dtype=float),
                "pos_units": pd.DataFrame(),
                "turnover_units": pd.DataFrame(),
                "cost": pd.Series(dtype=float)}

    # Align to common index
    idx = spread_px.index.intersection(signal.index)
    sp = spread_px[common_cols].reindex(idx).ffill()
    sig = signal[common_cols].reindex(idx).ffill().fillna(0.0)

    # 1. Spread changes (price units, NOT pct)
    dS = sp.diff().fillna(0.0)

    # 2. Vol estimate (price units)
    vol_daily = dS.ewm(span=vol_lookback, adjust=False).std()
    vol_daily = vol_daily.fillna(method="bfill")
    vol_ann_units = vol_daily * sqrt(252)
    # Prevent divide-by-zero: replace tiny values with median
    for col in vol_ann_units.columns:
        median_vol = vol_ann_units[col].median()
        if median_vol < 1e-8:
            median_vol = 1.0  # absolute fallback
        vol_ann_units[col] = vol_ann_units[col].where(
            vol_ann_units[col] > 1e-8, median_vol)

    # 3. Rebalance schedule: sample signals + vol weekly, shift by 1
    sig_reb = sig.resample(rebalance).last().shift(1)
    vol_reb = vol_ann_units.resample(rebalance).last().shift(1)
    vol_reb = vol_reb.fillna(method="bfill")

    # 4. Position units sizing
    pos_units_reb = sig_reb * (vol_target_annual * capital_base) / vol_reb
    pos_units = pos_units_reb.reindex(idx).ffill().fillna(0.0)

    # 5. PnL
    pnl_gross = (pos_units.shift(1) * dS).sum(axis=1)

    # 6. Transaction costs
    turnover_units = (pos_units - pos_units.shift(1)).abs()
    # TC per unit: bp × median spread level
    tc_per_unit = pd.DataFrame(index=idx, columns=common_cols, dtype=float)
    for col in common_cols:
        median_px = sp[col].abs().rolling(60, min_periods=5).median()
        median_px = median_px.fillna(sp[col].abs().median())
        tc_per_unit[col] = (tc_bp_per_side / 10_000.0) * median_px

    cost = (turnover_units * tc_per_unit).sum(axis=1).fillna(0.0)
    pnl_net = pnl_gross - cost

    # 7. Returns and equity
    ret_net = pnl_net / capital_base
    equity = (1 + ret_net).cumprod()
    equity.iloc[0] = 1.0

    # 8. Assertions
    non_finite_ret = (~np.isfinite(ret_net)).sum()
    non_finite_eq = (~np.isfinite(equity)).sum()
    if non_finite_ret > 0 or non_finite_eq > 0:
        bad_dates = ret_net[~np.isfinite(ret_net)].index.tolist()[:10]
        print(f"  !! {name}: {non_finite_ret} non-finite returns, "
              f"{non_finite_eq} non-finite equity")
        print(f"     Bad dates: {bad_dates}")
        # Force finite
        ret_net = ret_net.fillna(0.0).replace([np.inf, -np.inf], 0.0)
        equity = (1 + ret_net).cumprod()
        equity.iloc[0] = 1.0
    else:
        print(f"  {name}: OK — all finite. "
              f"Equity [{equity.iloc[0]:.4f} -> {equity.iloc[-1]:.4f}], "
              f"min={equity.min():.4f}")

    # Debug: worst 5 PnL days
    worst5 = ret_net.nsmallest(5)
    print(f"    Worst 5 days: " + ", ".join(
        f"{d:%Y-%m-%d}={v:.4%}" for d, v in worst5.items()))
    print(f"    NaNs in spread_px: {sp.isna().sum().sum()}, "
          f"NaNs in pos_units: {pos_units.isna().sum().sum()}")

    return {
        "ret_net": ret_net,
        "equity": equity * capital_base,
        "pos_units": pos_units,
        "turnover_units": turnover_units,
        "cost": cost,
    }


def compute_metrics(ret_series: pd.Series, name: str = "",
                    periods: int = 252) -> dict:
    """Compute standard performance metrics from a daily returns series.

    Handles vol=0 gracefully (Sharpe=0, warning).
    Never outputs inf/nan.
    """
    r = ret_series.dropna()
    if len(r) == 0:
        return {"Ann. Return": 0.0, "Ann. Vol": 0.0, "Sharpe": 0.0,
                "Max DD": 0.0, "Calmar": 0.0, "Hit Rate": 0.0,
                "Ann. Turnover": 0.0}

    n_years = len(r) / periods
    total = (1 + r).prod()
    ann_ret = total ** (1 / max(n_years, 0.01)) - 1
    ann_vol = r.std() * sqrt(periods)

    if ann_vol < 1e-10:
        sharpe = 0.0
        if name:
            print(f"  WARNING: {name} has zero vol — Sharpe set to 0")
    else:
        sharpe = ann_ret / ann_vol

    eq = (1 + r).cumprod()
    rm = eq.cummax()
    dd = (eq - rm) / rm
    max_dd = float(-dd.min()) if len(dd) > 0 else 0.0

    calmar = ann_ret / max_dd if max_dd > 1e-10 else 0.0
    hit = float((r > 0).sum() / len(r)) if len(r) > 0 else 0.0

    # Sanitise: replace any residual inf/nan with 0
    result = {"Ann. Return": ann_ret, "Ann. Vol": ann_vol, "Sharpe": sharpe,
              "Max DD": max_dd, "Calmar": calmar, "Hit Rate": hit}
    for k, v in result.items():
        if not np.isfinite(v):
            print(f"  WARNING: {name} metric {k} was {v}, set to 0")
            result[k] = 0.0
    return result


print("Backtest engines ready (Part C: PnL/capital spread backtest).")

## Results — Pure Carry vs Carry-with-Trend

Compare each sub-module both ways:
1. **Pure carry:** Signals without trend filter
2. **Carry-with-trend:** TSMOM sign used as soft filter (0.5× when misaligned)

Metals spreads and SOFR use the new PnL/capital backtest (no inf/nan).

In [ ]:
# Build SOFR slope series
sofr_slopes = pd.DataFrame(index=sofr_prices.index)
sofr_contracts = [c for c in sofr_cfg["instruments"] if c in sofr_prices.columns]
for i in range(len(sofr_contracts) - 1):
    sofr_slopes[f"SOFR_{i+1}v{i+2}"] = sofr_prices[sofr_contracts[i+1]] - sofr_prices[sofr_contracts[i]]
if len(sofr_contracts) >= 3:
    sofr_slopes["SOFR_fly"] = (sofr_prices[sofr_contracts[0]]
                               - 2*sofr_prices[sofr_contracts[1]]
                               + sofr_prices[sofr_contracts[2]])

# ── Run backtests: Pure vs Trend ─────────────────────────────────
bt_results = {}
metrics_rows = []

print("=" * 75)
print("  RUNNING BACKTESTS")
print("=" * 75)

# --- FX Carry (weights-based backtest) ---
for suffix, w in [("pure", fx_w_pure), ("trend", fx_w_trend)]:
    if w.empty:
        continue
    bt = backtest_weights(fx_spot_df, w, tc_bp=TC_BP)
    key = f"FX Carry ({suffix})"
    bt_results[key] = bt
    m = compute_metrics(bt["ret_net"], name=key)
    m["Strategy"] = key
    metrics_rows.append(m)

# --- Metals Spreads (new PnL/capital backtest) ---
for suffix, sig in [("pure", mt_sig_pure), ("trend", mt_sig_trend)]:
    if sig.empty:
        continue
    key = f"Metals Spreads ({suffix})"
    print(f"\n  {key}:")
    result = backtest_spread(
        metals_spreads, sig,
        vol_target_annual=TARGET_VOL, vol_lookback=30,
        rebalance=REBAL_FREQ, tc_bp_per_side=TC_BP,
        capital_base=1_000_000, name=key)
    if result["ret_net"].empty:
        continue
    bt_df = pd.DataFrame({
        "ret_net": result["ret_net"],
        "ret_gross": result["ret_net"] + result["cost"] / 1_000_000,
        "equity": result["equity"],
        "turnover": result["turnover_units"].sum(axis=1).fillna(0.0),
    })
    bt_results[key] = bt_df
    m = compute_metrics(result["ret_net"], name=key)
    m["Strategy"] = key
    metrics_rows.append(m)

# --- SOFR Curve (new PnL/capital backtest) ---
for suffix, sig in [("pure", sf_sig_pure), ("trend", sf_sig_trend)]:
    if sig.empty:
        key = f"SOFR Curve ({suffix})"
        print(f"\n  {key}: SKIPPED (module inactive)")
        continue
    key = f"SOFR Curve ({suffix})"
    print(f"\n  {key}:")
    result = backtest_spread(
        sofr_slopes, sig,
        vol_target_annual=TARGET_VOL, vol_lookback=30,
        rebalance=REBAL_FREQ, tc_bp_per_side=TC_BP,
        capital_base=1_000_000, name=key)
    if result["ret_net"].empty:
        continue
    bt_df = pd.DataFrame({
        "ret_net": result["ret_net"],
        "ret_gross": result["ret_net"] + result["cost"] / 1_000_000,
        "equity": result["equity"],
        "turnover": result["turnover_units"].sum(axis=1).fillna(0.0),
    })
    bt_results[key] = bt_df
    m = compute_metrics(result["ret_net"], name=key)
    m["Strategy"] = key
    metrics_rows.append(m)

# ── Results table ─────────────────────────────────────────────────
comp_df = pd.DataFrame(metrics_rows).set_index("Strategy")

fmt_comp = comp_df.copy()
for c in ["Ann. Return", "Ann. Vol", "Max DD", "Hit Rate"]:
    if c in fmt_comp.columns:
        fmt_comp[c] = fmt_comp[c].map("{:.1%}".format)
fmt_comp["Sharpe"] = fmt_comp["Sharpe"].map("{:.2f}".format)
fmt_comp["Calmar"] = fmt_comp["Calmar"].map("{:.2f}".format)

print("\n" + "=" * 75)
print("  PURE CARRY vs CARRY-WITH-TREND")
print("=" * 75)
display(fmt_comp)

# ── Trend filter impact ──────────────────────────────────────────
print("\nTrend filter impact:")
for mod in ["FX Carry", "Metals Spreads", "SOFR Curve"]:
    pure_key = f"{mod} (pure)"
    trend_key = f"{mod} (trend)"
    if pure_key in comp_df.index and trend_key in comp_df.index:
        d_sharpe = comp_df.loc[trend_key, "Sharpe"] - comp_df.loc[pure_key, "Sharpe"]
        d_dd = comp_df.loc[trend_key, "Max DD"] - comp_df.loc[pure_key, "Max DD"]
        print(f"  {mod:20s}  Sharpe delta: {d_sharpe:+.2f}  MaxDD delta: {d_dd:+.1%}")
        if abs(d_sharpe) < 0.001 and abs(d_dd) < 0.001:
            print(f"    ** WARNING: Pure and trend results are IDENTICAL. "
                  f"Trend filter may not be wired correctly.")
    elif trend_key not in comp_df.index:
        print(f"  {mod:20s}  (trend variant not available)")

# ── TSMOM correlation ────────────────────────────────────────────
import glob
tsmom_eq_files = sorted(glob.glob(str(Path("../outputs") / "tsmom_portfolio_equity_*.csv")))
if tsmom_eq_files:
    tsmom_port = pd.read_csv(tsmom_eq_files[-1], index_col=0, parse_dates=True)
    if "portfolio_ret_net" in tsmom_port.columns:
        tsmom_ret = tsmom_port["portfolio_ret_net"]
        print("\nCorrelation with TSMOM sleeve:")
        for key, bt in bt_results.items():
            if "trend" in key:
                corr = bt["ret_net"].corr(tsmom_ret.reindex(bt.index))
                print(f"  {key:30s}  rho = {corr:.3f}")
else:
    print("\nTSMOM equity file not found — skipping correlation check.")

## DEBUG — Submodule Return Integrity

For each submodule return series: count non-finite values, ann vol,
max absolute daily return, top 10 absolute daily returns.

Export: `outputs/debug_submodule_return_integrity.csv`

In [ ]:
integrity_rows = []
print("=" * 70)
print("  DEBUG — SUBMODULE RETURN INTEGRITY")
print("=" * 70)

for key, bt in bt_results.items():
    r = bt["ret_net"]
    n_nonfinite = (~np.isfinite(r)).sum()
    n_nan = r.isna().sum()
    ann_vol = r.std() * sqrt(252) if len(r) > 0 else 0.0
    max_abs = r.abs().max() if len(r) > 0 else 0.0

    print(f"\n  {key}:")
    print(f"    Obs: {len(r)}, Non-finite: {n_nonfinite}, NaN: {n_nan}")
    print(f"    Ann. vol: {ann_vol:.2%}, Max |daily ret|: {max_abs:.4%}")

    # Top 10 abs returns
    top10 = r.abs().nlargest(10)
    print(f"    Top 10 |daily ret|:")
    for d, v in top10.items():
        print(f"      {d:%Y-%m-%d}  {v:.4%}  (sign: {'+' if r.loc[d] > 0 else '-'})")

    integrity_rows.append({
        "strategy": key, "n_obs": len(r),
        "n_nonfinite": n_nonfinite, "n_nan": n_nan,
        "ann_vol": round(ann_vol, 4),
        "max_abs_ret": round(max_abs, 6),
    })

integrity_df = pd.DataFrame(integrity_rows)
integrity_df.to_csv(output_dir / "debug_submodule_return_integrity.csv", index=False)
print(f"\n  Exported: debug_submodule_return_integrity.csv")

## DEBUG — Spread Plumbing Check

For each metals spread: min/max spread level, min/max dS,
position units stats, and a detailed GC_1v2 diagnostic plot.

Export: `outputs/debug_spread_plumbing_gc1v2.csv`

In [ ]:
print("=" * 70)
print("  DEBUG — SPREAD PLUMBING CHECK")
print("=" * 70)

# Get position units from the metals trend backtest
mt_bt_result = backtest_spread(
    metals_spreads, mt_sig_trend,
    vol_target_annual=TARGET_VOL, vol_lookback=30,
    rebalance=REBAL_FREQ, tc_bp_per_side=TC_BP,
    capital_base=1_000_000, name="Metals (plumbing check)")
mt_pos = mt_bt_result["pos_units"]

for col in metals_spreads.columns:
    sp = metals_spreads[col].dropna()
    dS = sp.diff().dropna()

    print(f"\n  {col}:")
    print(f"    Spread level : min={sp.min():.2f}, max={sp.max():.2f}, "
          f"mean={sp.mean():.2f}, last={sp.iloc[-1]:.2f}")
    print(f"    dS           : min={dS.min():.2f}, max={dS.max():.2f}, "
          f"std={dS.std():.2f}")
    if col in mt_pos.columns:
        pos = mt_pos[col]
        print(f"    pos_units    : min={pos.min():.1f}, max={pos.max():.1f}, "
              f"mean={pos.mean():.1f}")
        # Count zero-crossing in spread
        zero_cross = ((sp.shift(1) * sp) < 0).sum()
        print(f"    Spread zero-crossings: {zero_cross}")

# GC_1v2 detailed diagnostic
if "GC_1v2" in metals_spreads.columns and "GC_1v2" in mt_pos.columns:
    gc_diag = pd.DataFrame({
        "spread_level": metals_spreads["GC_1v2"],
        "signal": mt_sig_trend["GC_1v2"] if "GC_1v2" in mt_sig_trend.columns
                  else np.nan,
        "pos_units": mt_pos["GC_1v2"],
    }).dropna()
    gc_diag.to_csv(output_dir / "debug_spread_plumbing_gc1v2.csv")

    # Plot
    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=True,
        row_heights=[0.6, 0.4],
        subplot_titles=["GC_1v2 Spread Level", "Position Units"],
    )
    fig.add_trace(go.Scatter(
        x=gc_diag.index, y=gc_diag["spread_level"],
        name="Spread", line=dict(color="#3498db", width=1.5),
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=gc_diag.index, y=gc_diag["pos_units"],
        name="Pos Units", line=dict(color="#e74c3c", width=1.5),
    ), row=2, col=1)
    fig.add_hline(y=0, line_dash="dash", line_color="gray", row=1, col=1)
    fig.add_hline(y=0, line_dash="dash", line_color="gray", row=2, col=1)
    fig.update_layout(title="DEBUG: GC_1v2 Spread + Position Units",
                      template="plotly_white", height=500)
    fig.show()
    print(f"  Exported: debug_spread_plumbing_gc1v2.csv")

## DEBUG — Trend Filter Activity (FX)

For each FX pair, compute the fraction of times the trend filter
actually changes the carry position (from pure to trend-filtered).

Export: `outputs/debug_fx_trend_filter_activity.csv`

In [ ]:
print("=" * 70)
print("  DEBUG — TREND FILTER ACTIVITY (FX)")
print("=" * 70)

tf_rows = []
for col in fx_w_pure.columns:
    if col not in fx_w_trend.columns:
        continue
    pure = fx_w_pure[col]
    trend = fx_w_trend[col]
    diff = (pure - trend).abs()

    n_active = (pure.abs() > 1e-10).sum()
    n_changed = (diff > 1e-10).sum()
    pct_changed = n_changed / max(n_active, 1) * 100

    # Average abs weight: pure vs trend
    avg_pure = pure.abs().mean()
    avg_trend = trend.abs().mean()

    print(f"  {col:12s}  active={n_active:>5d}  changed={n_changed:>5d}  "
          f"({pct_changed:5.1f}%)  avg|w| pure={avg_pure:.4f} trend={avg_trend:.4f}")

    tf_rows.append({
        "pair": col, "n_active": n_active, "n_changed": n_changed,
        "pct_changed": round(pct_changed, 1),
        "avg_abs_weight_pure": round(avg_pure, 4),
        "avg_abs_weight_trend": round(avg_trend, 4),
    })

tf_df = pd.DataFrame(tf_rows)
tf_df.to_csv(output_dir / "debug_fx_trend_filter_activity.csv", index=False)

# Check: if all pct_changed < 1%, warn
if all(r["pct_changed"] < 1.0 for r in tf_rows):
    print("\n  ** WARNING: Trend filter modified < 1% of positions for ALL pairs.")
    print("     This suggests the filter is not effectively wired.")
    print("     Check: are trend columns aligned? Are carry and trend on same quote convention?")
elif any(r["pct_changed"] > 0 for r in tf_rows):
    avg_pct = np.mean([r["pct_changed"] for r in tf_rows])
    print(f"\n  Average modification rate: {avg_pct:.1f}%")

print(f"  Exported: debug_fx_trend_filter_activity.csv")

## DEBUG — Turnover Spikes

Compute turnover series for each submodule and combined sleeve.
Identify top 20 turnover days and check overlap with rebalance dates.

Export: `outputs/debug_turnover_spikes.csv`

In [ ]:
print("=" * 70)
print("  DEBUG — TURNOVER SPIKES")
print("=" * 70)

# Collect turnover from each submodule
turnover_all = pd.DataFrame()
for key, bt in bt_results.items():
    if "trend" not in key:
        continue
    turnover_all[key] = bt["turnover"]

if not turnover_all.empty:
    turnover_all["combined"] = turnover_all.sum(axis=1)

    # Top 20 turnover days
    top20 = turnover_all["combined"].nlargest(20)

    # Generate rebalance dates for the period
    idx = turnover_all.index
    rebal_dates = pd.date_range(idx[0], idx[-1], freq=REBAL_FREQ)

    spike_rows = []
    print(f"\n  Top 20 turnover days:")
    print(f"  {'Date':12s}  {'Turnover':>10s}  {'On rebal?':>10s}  Breakdown")
    print(f"  {'-'*60}")
    for d, v in top20.items():
        on_rebal = "YES" if d in rebal_dates else "no"
        breakdown = "  ".join(
            f"{c}={turnover_all.loc[d, c]:.4f}"
            for c in turnover_all.columns if c != "combined"
        )
        print(f"  {d:%Y-%m-%d}  {v:10.4f}  {on_rebal:>10s}  {breakdown}")
        spike_rows.append({
            "date": d.strftime("%Y-%m-%d"), "combined_turnover": round(v, 6),
            "on_rebalance_date": on_rebal,
        })

    # Check: how many of top 20 are NOT on rebalance dates
    off_rebal = sum(1 for r in spike_rows if r["on_rebalance_date"] == "no")
    print(f"\n  {off_rebal}/20 top turnover days are NOT on rebalance dates.")
    if off_rebal > 5:
        print("  ** WARNING: Significant turnover outside rebalance dates. "
              "Check that weights are properly frozen.")

    spike_df = pd.DataFrame(spike_rows)
    spike_df.to_csv(output_dir / "debug_turnover_spikes.csv", index=False)
    print(f"  Exported: debug_turnover_spikes.csv")
else:
    print("  No turnover data available.")

## Combined Carry Sleeve

Combine FX + Metals + SOFR carry sub-modules (carry-with-trend variants)
using equal risk weighting, then apply portfolio-level vol targeting overlay
to 10% annual (sampled weekly, frozen within week).

If SOFR is inactive, weight redistributes to remaining modules
(if `redistribute_inactive_module=true`) or stays as cash.

In [ ]:
# ── Collect sub-module returns ─────────────────────────────────────
sub_returns = {}

# FX carry
if "FX Carry (trend)" in bt_results:
    sub_returns["fx"] = bt_results["FX Carry (trend)"]["ret_net"]

# Metals spreads
if "Metals Spreads (trend)" in bt_results:
    sub_returns["metals"] = bt_results["Metals Spreads (trend)"]["ret_net"]

# SOFR curve (only if active)
if SOFR_MODULE_ACTIVE and "SOFR Curve (trend)" in bt_results:
    sub_returns["sofr"] = bt_results["SOFR Curve (trend)"]["ret_net"]

n_modules = len(sub_returns)
print(f"Active carry sub-modules: {n_modules} — {list(sub_returns.keys())}")

if not sub_returns:
    print("No sub-module returns available!")
else:
    sub_ret_df = pd.DataFrame(sub_returns).fillna(0.0)

    # Equal risk weight across active modules
    if REDISTRIBUTE_INACTIVE and n_modules < 3:
        print(f"  Redistributing: {n_modules} active modules share full budget")
    module_weight = 1.0 / n_modules
    carry_ret_raw = (sub_ret_df * module_weight).sum(axis=1)

    # ── Portfolio vol targeting overlay (weekly frozen) ───────────
    carry_vol_rolling = carry_ret_raw.rolling(60, min_periods=20).std() * sqrt(252)
    scale = (TARGET_VOL / carry_vol_rolling.replace(0, np.nan)).clip(0.5, 2.0).fillna(1.0)

    # Freeze weekly — overlay scale only changes at rebalance
    scale_weekly = scale.resample(REBAL_FREQ).last()
    scale_daily = scale_weekly.reindex(carry_ret_raw.index).ffill().fillna(1.0)

    carry_ret_scaled = carry_ret_raw * scale_daily
    carry_equity = 1_000_000 * (1 + carry_ret_scaled).cumprod()

    # Combined turnover
    carry_turnover = pd.Series(0.0, index=carry_ret_raw.index)
    for key, bt in bt_results.items():
        if "trend" in key:
            carry_turnover += bt["turnover"].reindex(carry_turnover.index).fillna(0)

    # Metrics
    carry_sleeve_metrics = compute_metrics(carry_ret_scaled, name="COMBINED CARRY SLEEVE")

    # Realised vol check
    carry_vol_post = carry_ret_scaled.rolling(60, min_periods=20).std() * sqrt(252)

    print("\n" + "=" * 65)
    print("  COMBINED CARRY SLEEVE METRICS (IS 2015-2022)")
    print("=" * 65)
    for k, v in carry_sleeve_metrics.items():
        if isinstance(v, float):
            if "Return" in k or "Vol" in k or "DD" in k or "Rate" in k:
                print(f"  {k:20s}: {v:.1%}")
            elif "Turnover" in k:
                print(f"  {k:20s}: {v:.1f}x")
            else:
                print(f"  {k:20s}: {v:.2f}")

    print(f"\n  Realised vol (post-overlay):")
    print(f"    Mean   : {carry_vol_post.dropna().mean():.1%}  (target: {TARGET_VOL:.0%})")
    print(f"    Median : {carry_vol_post.dropna().median():.1%}")
    print(f"    Range  : [{carry_vol_post.dropna().min():.1%}, {carry_vol_post.dropna().max():.1%}]")

    # Overlay scale diagnostic
    print(f"\n  Vol overlay scale (weekly frozen):")
    print(f"    Mean   : {scale_daily.mean():.3f}")
    print(f"    Range  : [{scale_daily.min():.3f}, {scale_daily.max():.3f}]")
    changes_per_year = (scale_daily.diff().abs() > 1e-6).sum() / (len(scale_daily)/252)
    print(f"    Changes: {changes_per_year:.0f}/year (expect ~52)")

## Visualisations

1. Equity curves by sub-module + combined carry sleeve
2. FX carry ranking heatmap
3. Metals spread z-score heatmap
4. SOFR slope + signal panel (if active)
5. Turnover (weekly spikes)

In [ ]:
# --- 1. Equity Curves ---
fig_eq = go.Figure()
colors = {"FX Carry (trend)": "#3498db", "Metals Spreads (trend)": "#f39c12",
          "SOFR Curve (trend)": "#2ecc71"}
for key, bt in bt_results.items():
    if "trend" not in key:
        continue
    eq = bt["equity"]
    eq_norm = eq / eq.iloc[0]
    fig_eq.add_trace(go.Scatter(
        x=eq_norm.index, y=eq_norm, name=key,
        line=dict(color=colors.get(key, "#95a5a6"), width=1.5),
    ))
if "carry_equity" in dir():
    eq_norm = carry_equity / carry_equity.iloc[0]
    fig_eq.add_trace(go.Scatter(
        x=eq_norm.index, y=eq_norm, name="Combined Sleeve",
        line=dict(color="#2c3e50", width=2.5),
    ))
fig_eq.update_layout(
    title="Carry Strategy — Equity Curves (IS: 2015-2022, carry-with-trend)",
    template="plotly_white", height=500, hovermode="x unified",
    legend=dict(orientation="h", y=-0.15),
    yaxis_title="Growth of $1", yaxis_tickformat="$.2f",
)
fig_eq.show()

# --- 2. FX Carry Ranking Heatmap ---
carry_matrix = results["fx"]["carry_matrix"]
if not carry_matrix.empty:
    carry_monthly = (carry_matrix * 10_000).resample("ME").last().dropna()
    fig_fx = px.imshow(
        carry_monthly.T.round(0),
        title="FX Implied Carry (bp annualised) — Monthly",
        labels={"x": "", "y": "Pair", "color": "Carry (bp)"},
        color_continuous_scale="RdYlGn", aspect="auto",
    )
    fig_fx.update_layout(template="plotly_white", height=300,
                         xaxis=dict(dtick="M6", tickformat="%Y-%m"))
    fig_fx.show()

# --- 3. Metals Spread Z-Score Heatmap ---
metals_z = results["metals"]["zscores"]
if not metals_z.empty:
    zs_monthly = metals_z.resample("ME").last().dropna()
    fig_mz = px.imshow(
        zs_monthly.T.round(2),
        title="Metals Calendar Spread Z-Scores — Monthly",
        labels={"x": "", "y": "Spread", "color": "Z-Score"},
        color_continuous_scale="RdBu_r", zmin=-3, zmax=3, aspect="auto",
    )
    fig_mz.add_annotation(
        text=f"Entry: +/-{met_cfg['entry_zscore']}s  |  Exit: +/-{met_cfg['exit_zscore']}s",
        xref="paper", yref="paper", x=0.01, y=1.08,
        showarrow=False, font=dict(size=11, color="gray"),
    )
    fig_mz.update_layout(template="plotly_white", height=300,
                         xaxis=dict(dtick="M6", tickformat="%Y-%m"))
    fig_mz.show()

# --- 4. SOFR Slope + Signal (only if active) ---
if SOFR_MODULE_ACTIVE:
    sofr_sigs = results["sofr"]["signals"]
    sofr_sl   = results["sofr"]["slopes"]
    if not sofr_sl.empty:
        fig_sofr = make_subplots(
            rows=2, cols=1, shared_xaxes=True,
            row_heights=[0.6, 0.4],
            subplot_titles=["SOFR Slopes (price space)", "Signals"],
            vertical_spacing=0.10,
        )
        for col in sofr_sl.columns:
            fig_sofr.add_trace(go.Scatter(
                x=sofr_sl.index, y=sofr_sl[col], name=col, line=dict(width=1.5),
            ), row=1, col=1)
        for col in sofr_sigs.columns:
            fig_sofr.add_trace(go.Scatter(
                x=sofr_sigs.index, y=sofr_sigs[col], name=f"{col} sig",
                line=dict(width=1, dash="dot"),
            ), row=2, col=1)
        fig_sofr.update_layout(
            title="SOFR Curve — Slopes & Signals", template="plotly_white",
            height=550, hovermode="x unified",
            legend=dict(orientation="h", y=-0.15),
        )
        fig_sofr.update_yaxes(title_text="Slope", row=1, col=1)
        fig_sofr.update_yaxes(title_text="Signal", range=[-1.2, 1.2], row=2, col=1)
        fig_sofr.show()
else:
    print("SOFR module inactive — skipping SOFR chart.")

# --- 5. Turnover ---
if bt_results:
    total_turn = pd.Series(0.0, dtype=float)
    for key, bt in bt_results.items():
        if "trend" in key:
            total_turn = total_turn.add(bt["turnover"], fill_value=0.0)
    fig_turn = go.Figure()
    fig_turn.add_trace(go.Bar(
        x=total_turn.index, y=total_turn,
        marker_color="#3498db", opacity=0.7, name="Turnover",
    ))
    fig_turn.update_layout(
        title="Carry Sleeve — Daily Turnover (expect weekly spikes)",
        template="plotly_white", height=300,
        yaxis_title="Turnover |Dw|", hovermode="x unified",
    )
    fig_turn.show()

## Export

Save signals, equity curves, and summary to `outputs/`.

In [ ]:
datestamp = datetime.now().strftime("%Y%m%d")

# FX carry signals
p1 = output_dir / f"carry_fx_signals_{datestamp}.csv"
results["fx"]["signals"].to_csv(p1)

# Metals spread signals
p2 = output_dir / f"carry_metals_spreads_signals_{datestamp}.csv"
results["metals"]["signals"].to_csv(p2)

# SOFR signals
p3 = output_dir / f"carry_sofr_signals_{datestamp}.csv"
if not results["sofr"]["signals"].empty:
    results["sofr"]["signals"].to_csv(p3)
else:
    # Write empty placeholder with status
    with open(p3, "w") as f:
        f.write(f"# SOFR module status: {results['sofr']['status']}\n")

# Equity curves
eq_export = pd.DataFrame()
for key, bt in bt_results.items():
    if "trend" in key:
        eq_export[key] = bt["equity"]
if "carry_equity" in dir():
    eq_export["Combined Sleeve"] = carry_equity
p4 = output_dir / f"carry_equity_curves_{datestamp}.csv"
eq_export.to_csv(p4)

# Carry sleeve daily returns (for portfolio integration)
p5 = output_dir / f"carry_sleeve_daily_returns_{datestamp}.csv"
if "carry_ret_scaled" in dir():
    carry_ret_scaled.to_csv(p5, header=["carry_sleeve_ret_net"])

# Summary HTML
p6 = output_dir / f"carry_summary_{datestamp}.html"
html = (
    "<h2>Carry Strategy — IS Performance (2015-2022)</h2>\n"
    "<p>FX cross-sectional carry + Metals calendar spreads + SOFR curve<br>"
    f"Trend filter: {TREND_FILTER_MODE} | Vol target: {TARGET_VOL:.0%} | TC: {TC_BP:.0f}bp<br>"
    f"SOFR status: {results['sofr']['status']}<br>"
    f"Spread convention: {SPREAD_CONVENTION}</p>\n"
    "<h3>Pure Carry vs Carry-with-Trend</h3>\n"
    + fmt_comp.to_html()
)
if "carry_sleeve_metrics" in dir():
    html += "<br><h3>Combined Carry Sleeve</h3>\n<pre>"
    for k, v in carry_sleeve_metrics.items():
        if isinstance(v, float):
            html += f"  {k:20s}: {v:.4f}\n"
    html += "</pre>"
with open(p6, "w") as f:
    f.write(html)

print(f"Exported to {output_dir.resolve()}/")
for p in [p1, p2, p3, p4, p5, p6]:
    print(f"  {p.name}")

# Also list debug exports from earlier cells
debug_files = [
    "carry_data_coverage.csv",
    "debug_spread_convention_sample.csv",
    "debug_submodule_return_integrity.csv",
    "debug_spread_plumbing_gc1v2.csv",
    "debug_fx_trend_filter_activity.csv",
    "debug_turnover_spikes.csv",
]
print(f"\nDebug exports:")
for f in debug_files:
    print(f"  {f}")

print(f"\nNotebook complete: {datetime.now():%Y-%m-%d %H:%M}")